# Rolling update evaluation

**Mostly Claude generated** Verified and tested for validity and accuracy of presented data.

Input is `rolling_update_test.csv`, produced by `probe-availability.sh` (one probe every ~2s).
Set `DETECTED_AT` to the timestamp the operator logged the new version.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

CSV_PATH = "rolling_update_test.csv"
DETECTED_AT = "2026-07-13T12:53:33Z"  # operator log: new version detected

df = pd.read_csv(CSV_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# probe start = first sample's wall clock minus its own elapsed offset,
# so elapsed_s and the operator's log timestamps share one origin.
probe_start = df["timestamp"].iloc[0] - pd.Timedelta(seconds=int(df["elapsed_s"].iloc[0]))
detected_s = (pd.to_datetime(DETECTED_AT) - probe_start).total_seconds()

ok = df[df["success"] == 1]
fail = df[df["success"] == 0]

# A failed probe's response_time is the time to *refuse*, not a latency - keep it off the line.
lat = df["response_time_s"].where(df["success"] == 1)

# Sampling bounds the outage: it began after the last good probe and ended by the next one.
first_fail = fail["elapsed_s"].min()
last_fail = fail["elapsed_s"].max()
last_ok_pre = ok[ok["elapsed_s"] < first_fail]["elapsed_s"].max()
first_ok_post = ok[ok["elapsed_s"] > last_fail]["elapsed_s"].min()

downtime_max = first_ok_post - last_ok_pre
downtime_est = len(fail) * df["elapsed_s"].diff().median()

print(f"probes            : {len(df)}  ({len(fail)} failed)")
print(f"detect -> outage  : {first_fail - detected_s:.0f} s")
print(f"downtime          : ~{downtime_est:.0f} s  (bounded: >0 and <= {downtime_max:.0f} s)")
print(f"latency p50 / max : {ok['response_time_s'].median():.3f} s / {ok['response_time_s'].max():.3f} s")

In [ ]:
BLUE = "#2a78d6"      # latency series
GOOD = "#0ca30c"      # probe succeeded
CRITICAL = "#d03b3b"  # probe failed
SURFACE = "#fcfcfb"
PRIMARY = "#0b0b0b"
SECONDARY = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 10,
    "text.color": PRIMARY, "axes.labelcolor": SECONDARY,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
})

In [ ]:
fig, (ax, strip) = plt.subplots(
    2, 1, figsize=(12, 6), sharex=True,
    gridspec_kw={"height_ratios": [4, 1], "hspace": 0.15},
)

# outage window: shaded from last good probe to first good probe after
ax.axvspan(last_ok_pre, first_ok_post, color=CRITICAL, alpha=0.10, lw=0, zorder=1)
strip.axvspan(last_ok_pre, first_ok_post, color=CRITICAL, alpha=0.10, lw=0, zorder=1)

# operator event marker
for a in (ax, strip):
    a.axvline(detected_s, color=SECONDARY, lw=1.5, ls=(0, (4, 3)), zorder=2)

# latency line (single series -> no legend; the title names it)
ax.plot(df["elapsed_s"], lat, color=BLUE, lw=2,
        solid_capstyle="round", solid_joinstyle="round", zorder=3)

# selective direct labels: the peak and the outage, nothing else
peak = ok.loc[ok["response_time_s"].idxmax()]
ax.plot(peak["elapsed_s"], peak["response_time_s"], "o", ms=8, color=BLUE,
        mec=SURFACE, mew=2, zorder=4)
ax.annotate(f"{peak['response_time_s']:.3f} s",
            (peak["elapsed_s"], peak["response_time_s"]),
            textcoords="offset points", xytext=(0, 12),
            ha="center", color=SECONDARY, fontsize=9)

ax.set_ylabel("response time (s)")
ax.set_ylim(0, ok["response_time_s"].max() * 1.45)

ax.annotate("new version\ndetected", (detected_s, ax.get_ylim()[1]),
            textcoords="offset points", xytext=(6, -6),
            ha="left", va="top", color=SECONDARY, fontsize=9)

ax.annotate(f"connection refused\n{len(fail)} probe \u00b7 downtime \u2264 {downtime_max:.0f}s",
            ((last_ok_pre + first_ok_post) / 2, ax.get_ylim()[1] * 0.72),
            textcoords="offset points", xytext=(14, 0),
            ha="left", va="center", color=CRITICAL, fontsize=9,
            arrowprops=dict(arrowstyle="-", color=CRITICAL, lw=1, shrinkA=0, shrinkB=2))

ax.grid(axis="y", color=GRID, lw=0.8, ls="-", zorder=0)
ax.set_axisbelow(True)
ax.spines["left"].set_color(BASELINE)
ax.spines["bottom"].set_color(BASELINE)

ax.set_title("Front-app availability across a rolling update",
             loc="left", fontsize=13, fontweight="semibold", pad=28)
ax.text(0, 1.06, f"POST /sbom every ~2s \u00b7 {len(df)} probes \u00b7 {len(fail)} failed",
        transform=ax.transAxes, color=SECONDARY, fontsize=10)

# wall clock on top, so rows line up with operator logs (same scale, relabelled)
secax = ax.secondary_xaxis("top")
secax.xaxis.set_major_formatter(FuncFormatter(
    lambda e, _: (probe_start + pd.Timedelta(seconds=e)).strftime("%H:%M:%S")))
secax.tick_params(colors=MUTED)
secax.spines["top"].set_visible(False)

# availability strip: status by exception, one tick per probe
colors = [GOOD if s == 1 else CRITICAL for s in df["success"]]
strip.bar(df["elapsed_s"], 1, width=1.6, color=colors, align="center", zorder=3)
strip.set_ylim(0, 1)
strip.set_yticks([])
strip.set_xlabel("seconds since probe start")
strip.spines["left"].set_visible(False)
strip.spines["bottom"].set_color(BASELINE)
strip.legend(
    handles=[Line2D([], [], marker="s", ls="", ms=8, color=GOOD, label="200 OK"),
             Line2D([], [], marker="s", ls="", ms=8, color=CRITICAL, label="failed (curl 7)")],
    loc="upper left", bbox_to_anchor=(0, -0.55), ncol=2,
    frameon=False, fontsize=9, handletextpad=0.4, columnspacing=1.2,
)

fig.savefig("rolling_update_availability.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
plt.show()

## Reading the result

**Downtime is bounded** One failed probe at a ~2s sampling
shows that the outage began after the last good probe and was over by the next one, so
downtime lies in `(0s, 4s]` with ~2s as the point estimate.

**Detection to outage is ~65s** - the image pull plus the rollout takes time.

**The latency peak lands two seconds before the operator logged detection.** Polling and pulling may be cause. Possibly due to locally running operator sharing network access with probe test.